# Model Registry & CI/CD: Automating Safe Model Deployment

## Learning Objectives
- Design automated model deployment pipelines
- Implement approval gates and safety checks
- Scale deployment to 100+ models
- Handle rollback scenarios

## Interview Questions This Covers
- "Model is trained. How do you safely deploy it?"
- "Accuracy dropped. Auto-rollback?"
- "Design approval workflow for 100 models"

## Basic: Model Registry + Versioning

In [ ]:
registry_example = '''
Model Registry Workflow:

STEP 1: Register model
  mlflow.models.log(model)
  Result: MLflow creates entry
    - Model ID: fraud-detector
    - Version: 1.0.0
    - Metrics: accuracy=0.95, precision=0.98
    - Status: staging

STEP 2: Validate in staging
  - Run shadow test on production data (1 week)
  - Compare predictions to baseline
  - If accuracy good: mark as ready

STEP 3: Request approval
  - Create review request
  - Require 2 senior engineers
  - Check: metrics passing? Documentation complete? Risks identified?

STEP 4: Approve and promote
  - Status: staging → approved
  - Schedule canary deployment

STEP 5: Canary
  - 5% traffic on new model
  - Monitor for 24h
  - If good: expand. If bad: rollback instantly.

STEP 6: Archive previous version
  - Status: production → archived
  - Keep for history and rollback
'''

print("MODEL REGISTRY WORKFLOW")
print()
print(registry_example)

## Advanced: CI/CD Pipeline with Approval Gates

In [ ]:
cicd_pipeline = '''
GitHub Actions CI/CD Pipeline for Model Deployment

on: [push]

jobs:
  train:
    runs-on: gpu-runner
    steps:
      - name: Train model
        run: python train.py
      - name: Validate accuracy
        run: python validate.py
        # Fail if accuracy < threshold
  
  validate:
    needs: train
    steps:
      - name: Unit tests
        run: pytest tests/
      - name: Integration tests
        run: python integration_test.py
      - name: Shadow test
        run: python shadow_test.py
        # Compare to baseline on 1 week production data
  
  build:
    needs: validate
    steps:
      - name: Create Docker image
        run: docker build -t fraud:abc123 .
      - name: Push to registry
        run: docker push myregistry.com/fraud:abc123
  
  approval:
    needs: build
    runs-on: ubuntu-latest
    steps:
      - name: Create review request
        run: gh pr create --body "Ready for approval"
      - name: Wait for approval
        run: gh pr status --checks
        # Blocks until 2 approvals
  
  canary:
    needs: approval
    steps:
      - name: Deploy to 5% traffic
        run: kubectl set image fraud=myregistry/fraud:abc123
      - name: Monitor for 24h
        run: python monitor.py
      - name: If metrics good, expand
        run: kubectl apply -f 25-percent-rollout.yaml
'''

print("CI/CD PIPELINE WITH APPROVAL GATES")
print()
print(cicd_pipeline)
print()
print("Key features:")
print("- Train: automated, fails on bad accuracy")
print("- Validate: unit + integration + shadow tests")
print("- Build: Docker image tagged with commit hash")
print("- Approval: requires 2 human reviews")
print("- Canary: 5% traffic, 24h monitoring")

## Real-World Examples: Netflix, Stripe, Uber

In [ ]:
examples = '''
NETFLIX: Fully Automated Deployment
- 50+ models deployed per day
- Auto-approval if: accuracy > threshold, shadow test passed, fairness metrics ok
- Canary to 5%, expand on schedule (5% → 25% → 50% → 100%)
- Auto-rollback if: any metric degrades > 2%
- Result: 2 hours from training to production

STRIPE: Safety-First Workflow
- Manual approval required (safety is critical)
- Approval checklist: metrics passing? shadow test good? risks documented?
- Canary to 1% fraud traffic (conservative, low risk)
- 1-week shadow test before canary (extra validation)
- Auto-rollback if fraud detection rate < 95% or false decline rate > 1% increase

UBER: Scale-Optimized Pipeline
- 100+ models in registry
- Tiered automation:
  - Low-risk (bug fix): auto-approve, fast canary (6h not 24h)
  - Medium-risk (model change): manual approval, standard canary
  - High-risk (infrastructure): on-call engineer monitoring
- Result: balance speed (low-risk fast) with safety (high-risk careful)
'''

print(examples)

## Interview Case Study: Designing Registry & CI/CD

In [ ]:
case_study = '''
SCENARIO: Fraud model deployment for payment processor
- Current model: 96% detection rate
- New model: 97% detection rate (1% improvement)
- Risk: higher detection might increase false declines

DESIGN SAFE DEPLOYMENT:

1. TRAINING & VALIDATION
   - Train new model
   - Validate: accuracy > 96% ✓
   - Shadow test: run on 1 month production fraud data
     - Compare: detection 97%, false decline +0.5% (acceptable)
     - Result: passed

2. REGISTER & APPROVE
   - Register in MLflow: fraud-detector:v1.1.0
   - Create approval request with checklist:
     ✓ Accuracy improved
     ✓ Shadow test passed (detection 97%)
     ✓ False decline rate acceptable (+0.5% < +1% threshold)
     ✓ Risk assessment: medium (operational, not critical)
   - Require 2 approvals from senior engineers

3. CANARY DEPLOYMENT
   - Deploy to 1% of transaction processing
   - Monitor for 24 hours:
     - Detection rate: 97% (good)
     - False decline rate: +0.4% (within +1% threshold)
     - No errors
   - Auto-rollback threshold: detection < 96% or decline > +1%

4. GRADUAL ROLLOUT
   - 1% → 5% (24h) if metrics stable
   - 5% → 25% (24h) if metrics stable
   - 25% → 100% (after 72h total, all metrics stable)

5. POST-DEPLOYMENT
   - Continue monitoring
   - Archive previous version (v1.0.0) for rollback
   - Weekly review: detection rate trend, false decline trend

RESULT: Safe, auditable, reversible deployment
'''

print(case_study)

## Key Takeaways

**CI/CD transforms ML:** Automated testing + approval gates = safe iteration.

**Approval gates prevent bad models:** Not every model that passes tests should deploy.

**Canary deployment limits blast radius:** 1% affected is better than 100%.

**Common pattern:** Train → validate → build → approve → canary → gradual rollout → monitor → archive.